In [1]:
# ==========================================================
# CO4 - AT2 - QUESTION 31
# Motion Model-Based Animation of a Walking Character Rig
# ==========================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

# ==========================================================
# STEP 1: WALKING MODEL PARAMETERS
# ==========================================================

# Character dimensions in arbitrary units

TORSO_LENGTH = 2.5

THIGH_LENGTH = 1.8
SHIN_LENGTH = 1.8

UPPER_ARM_LENGTH = 1.2
FOREARM_LENGTH = 1.0

# Walking parameters

STEP_LENGTH = 2.0

WALKING_SPEED = 1.2

GAIT_FREQUENCY = 1.2

# Number of animation frames

NUM_FRAMES = 120

# ==========================================================
# STEP 2: TIME VECTOR
# ==========================================================

time_values = np.linspace(
    0,
    4,
    NUM_FRAMES
)

# ==========================================================
# STEP 3: MOTION MODEL
# ==========================================================

def walking_angles(t):

    # ------------------------------------------------------
    # Leg motion
    # ------------------------------------------------------

    # Left leg
    left_hip = (
        25 *
        np.sin(
            2 * np.pi *
            GAIT_FREQUENCY *
            t
        )
    )

    left_knee = (
        15 +
        25 *
        (
            1 -
            np.cos(
                2 * np.pi *
                GAIT_FREQUENCY *
                t
            )
        ) / 2
    )

    # Right leg is approximately opposite phase
    right_hip = (
        25 *
        np.sin(
            2 * np.pi *
            GAIT_FREQUENCY *
            t +
            np.pi
        )
    )

    right_knee = (
        15 +
        25 *
        (
            1 -
            np.cos(
                2 * np.pi *
                GAIT_FREQUENCY *
                t +
                np.pi
            )
        ) / 2
    )

    # ------------------------------------------------------
    # Arm motion
    # ------------------------------------------------------

    left_shoulder = (
        20 *
        np.sin(
            2 * np.pi *
            GAIT_FREQUENCY *
            t +
            np.pi
        )
    )

    right_shoulder = (
        20 *
        np.sin(
            2 * np.pi *
            GAIT_FREQUENCY *
            t
        )
    )

    return (
        np.radians(left_hip),
        np.radians(left_knee),
        np.radians(right_hip),
        np.radians(right_knee),
        np.radians(left_shoulder),
        np.radians(right_shoulder)
    )

# ==========================================================
# STEP 4: FORWARD KINEMATICS
# ==========================================================

def calculate_character(t):

    (
        left_hip_angle,
        left_knee_angle,
        right_hip_angle,
        right_knee_angle,
        left_arm_angle,
        right_arm_angle
    ) = walking_angles(t)

    # ------------------------------------------------------
    # Body position
    # ------------------------------------------------------

    pelvis_x = (
        WALKING_SPEED *
        t
    )

    pelvis_y = 4.2

    pelvis = np.array([
        pelvis_x,
        pelvis_y
    ])

    # ------------------------------------------------------
    # Head and torso
    # ------------------------------------------------------

    shoulder_center = pelvis + np.array([
        0,
        TORSO_LENGTH
    ])

    head = shoulder_center + np.array([
        0,
        1.0
    ])

    # ------------------------------------------------------
    # LEFT LEG
    # ------------------------------------------------------

    left_hip = pelvis + np.array([
        -0.35,
        0
    ])

    left_knee = left_hip + np.array([
        THIGH_LENGTH *
        np.sin(left_hip_angle),

        -THIGH_LENGTH *
        np.cos(left_hip_angle)
    ])

    left_foot = left_knee + np.array([
        SHIN_LENGTH *
        np.sin(
            left_hip_angle +
            left_knee_angle
        ),

        -SHIN_LENGTH *
        np.cos(
            left_hip_angle +
            left_knee_angle
        )
    ])

    # ------------------------------------------------------
    # RIGHT LEG
    # ------------------------------------------------------

    right_hip = pelvis + np.array([
        0.35,
        0
    ])

    right_knee = right_hip + np.array([
        THIGH_LENGTH *
        np.sin(right_hip_angle),

        -THIGH_LENGTH *
        np.cos(right_hip_angle)
    ])

    right_foot = right_knee + np.array([
        SHIN_LENGTH *
        np.sin(
            right_hip_angle +
            right_knee_angle
        ),

        -SHIN_LENGTH *
        np.cos(
            right_hip_angle +
            right_knee_angle
        )
    ])

    # ------------------------------------------------------
    # LEFT ARM
    # ------------------------------------------------------

    left_shoulder = shoulder_center + np.array([
        -0.6,
        0
    ])

    left_elbow = left_shoulder + np.array([
        UPPER_ARM_LENGTH *
        np.sin(left_arm_angle),

        -UPPER_ARM_LENGTH *
        np.cos(left_arm_angle)
    ])

    left_hand = left_elbow + np.array([
        FOREARM_LENGTH *
        np.sin(left_arm_angle),

        -FOREARM_LENGTH *
        np.cos(left_arm_angle)
    ])

    # ------------------------------------------------------
    # RIGHT ARM
    # ------------------------------------------------------

    right_shoulder = shoulder_center + np.array([
        0.6,
        0
    ])

    right_elbow = right_shoulder + np.array([
        UPPER_ARM_LENGTH *
        np.sin(right_arm_angle),

        -UPPER_ARM_LENGTH *
        np.cos(right_arm_angle)
    ])

    right_hand = right_elbow + np.array([
        FOREARM_LENGTH *
        np.sin(right_arm_angle),

        -FOREARM_LENGTH *
        np.cos(right_arm_angle)
    ])

    return {
        "pelvis": pelvis,
        "shoulder": shoulder_center,
        "head": head,

        "left_hip": left_hip,
        "left_knee": left_knee,
        "left_foot": left_foot,

        "right_hip": right_hip,
        "right_knee": right_knee,
        "right_foot": right_foot,

        "left_shoulder": left_shoulder,
        "left_elbow": left_elbow,
        "left_hand": left_hand,

        "right_shoulder": right_shoulder,
        "right_elbow": right_elbow,
        "right_hand": right_hand
    }

# ==========================================================
# STEP 5: CREATE FIGURE
# ==========================================================

fig, ax = plt.subplots(
    figsize=(8, 8)
)

ax.set_xlim(
    -3,
    8
)

ax.set_ylim(
    -1,
    8
)

ax.set_aspect(
    "equal"
)

ax.set_xlabel(
    "Horizontal Position"
)

ax.set_ylabel(
    "Vertical Position"
)

ax.set_title(
    "Motion Model-Based Walking Character Rig"
)

ax.grid(True)

# Ground
ax.axhline(
    0,
    linewidth=2
)

# ==========================================================
# STEP 6: ANIMATION UPDATE FUNCTION
# ==========================================================

def update(frame):

    ax.clear()

    ax.set_xlim(
        -3,
        8
    )

    ax.set_ylim(
        -1,
        8
    )

    ax.set_aspect(
        "equal"
    )

    ax.set_xlabel(
        "Horizontal Position"
    )

    ax.set_ylabel(
        "Vertical Position"
    )

    ax.set_title(
        "Motion Model-Based Walking Character Rig"
    )

    ax.grid(True)

    # Ground
    ax.axhline(
        0,
        linewidth=2
    )

    # Current time
    t = time_values[frame]

    points = calculate_character(t)

    # ------------------------------------------------------
    # LEFT LEG
    # ------------------------------------------------------

    ax.plot(
        [
            points["left_hip"][0],
            points["left_knee"][0],
            points["left_foot"][0]
        ],
        [
            points["left_hip"][1],
            points["left_knee"][1],
            points["left_foot"][1]
        ],
        linewidth=4
    )

    # ------------------------------------------------------
    # RIGHT LEG
    # ------------------------------------------------------

    ax.plot(
        [
            points["right_hip"][0],
            points["right_knee"][0],
            points["right_foot"][0]
        ],
        [
            points["right_hip"][1],
            points["right_knee"][1],
            points["right_foot"][1]
        ],
        linewidth=4
    )

    # ------------------------------------------------------
    # TORSO
    # ------------------------------------------------------

    ax.plot(
        [
            points["pelvis"][0],
            points["shoulder"][0]
        ],
        [
            points["pelvis"][1],
            points["shoulder"][1]
        ],
        linewidth=5
    )

    # ------------------------------------------------------
    # HEAD
    # ------------------------------------------------------

    head_circle = plt.Circle(
        (
            points["head"][0],
            points["head"][1]
        ),
        0.35,
        fill=False,
        linewidth=3
    )

    ax.add_patch(
        head_circle
    )

    # ------------------------------------------------------
    # LEFT ARM
    # ------------------------------------------------------

    ax.plot(
        [
            points["left_shoulder"][0],
            points["left_elbow"][0],
            points["left_hand"][0]
        ],
        [
            points["left_shoulder"][1],
            points["left_elbow"][1],
            points["left_hand"][1]
        ],
        linewidth=3
    )

    # ------------------------------------------------------
    # RIGHT ARM
    # ------------------------------------------------------

    ax.plot(
        [
            points["right_shoulder"][0],
            points["right_elbow"][0],
            points["right_hand"][0]
        ],
        [
            points["right_shoulder"][1],
            points["right_elbow"][1],
            points["right_hand"][1]
        ],
        linewidth=3
    )

    # ------------------------------------------------------
    # JOINTS
    # ------------------------------------------------------

    joint_names = [
        "pelvis",
        "left_hip",
        "left_knee",
        "right_hip",
        "right_knee",
        "left_shoulder",
        "left_elbow",
        "right_shoulder",
        "right_elbow"
    ]

    for name in joint_names:

        point = points[name]

        ax.scatter(
            point[0],
            point[1],
            s=50
        )

    # ------------------------------------------------------
    # Ground-contact feet
    # ------------------------------------------------------

    ax.scatter(
        points["left_foot"][0],
        0,
        s=60
    )

    ax.scatter(
        points["right_foot"][0],
        0,
        s=60
    )

    # ------------------------------------------------------
    # Information
    # ------------------------------------------------------

    ax.text(
        0.02,
        0.95,
        f"Time: {t:.2f} s",
        transform=ax.transAxes
    )

    ax.text(
        0.02,
        0.90,
        f"Walking speed: {WALKING_SPEED:.2f} units/s",
        transform=ax.transAxes
    )

# ==========================================================
# STEP 7: CREATE ANIMATION
# ==========================================================

animation = FuncAnimation(
    fig,
    update,
    frames=NUM_FRAMES,
    interval=40,
    repeat=True
)

plt.close(fig)

# Display animation
display(
    HTML(
        animation.to_jshtml()
    )
)

# ==========================================================
# STEP 8: MOTION METRICS
# ==========================================================

pelvis_positions = []

left_foot_positions = []

right_foot_positions = []

for t in time_values:

    character = calculate_character(t)

    pelvis_positions.append(
        character["pelvis"]
    )

    left_foot_positions.append(
        character["left_foot"]
    )

    right_foot_positions.append(
        character["right_foot"]
    )

pelvis_positions = np.array(
    pelvis_positions
)

left_foot_positions = np.array(
    left_foot_positions
)

right_foot_positions = np.array(
    right_foot_positions
)

# ----------------------------------------------------------
# Walking displacement
# ----------------------------------------------------------

total_displacement = (
    pelvis_positions[-1, 0]
    -
    pelvis_positions[0, 0]
)

# ----------------------------------------------------------
# Calculate estimated speed
# ----------------------------------------------------------

total_time = (
    time_values[-1]
    -
    time_values[0]
)

estimated_speed = (
    total_displacement /
    total_time
)

# ----------------------------------------------------------
# Calculate joint-angle range
# ----------------------------------------------------------

left_hip_angles = []

right_hip_angles = []

left_knee_angles = []

right_knee_angles = []

for t in time_values:

    angles = walking_angles(t)

    left_hip_angles.append(
        np.degrees(angles[0])
    )

    left_knee_angles.append(
        np.degrees(angles[1])
    )

    right_hip_angles.append(
        np.degrees(angles[2])
    )

    right_knee_angles.append(
        np.degrees(angles[3])
    )

left_hip_range = (
    max(left_hip_angles)
    -
    min(left_hip_angles)
)

right_hip_range = (
    max(right_hip_angles)
    -
    min(right_hip_angles)
)

left_knee_range = (
    max(left_knee_angles)
    -
    min(left_knee_angles)
)

right_knee_range = (
    max(right_knee_angles)
    -
    min(right_knee_angles)
)

# ==========================================================
# STEP 9: PRINT RESULTS
# ==========================================================

print(
    "\n=============================================="
)

print(
    "WALKING CHARACTER RIG RESULTS"
)

print(
    "=============================================="
)

print(
    "\nWalking speed parameter:",
    f"{WALKING_SPEED:.2f} units/s"
)

print(
    "Total walking displacement:",
    f"{total_displacement:.2f} units"
)

print(
    "Total animation duration:",
    f"{total_time:.2f} seconds"
)

print(
    "Estimated average speed:",
    f"{estimated_speed:.2f} units/s"
)

print(
    "\nJoint angle ranges:"
)

print(
    "Left hip:",
    f"{left_hip_range:.2f} degrees"
)

print(
    "Right hip:",
    f"{right_hip_range:.2f} degrees"
)

print(
    "Left knee:",
    f"{left_knee_range:.2f} degrees"
)

print(
    "Right knee:",
    f"{right_knee_range:.2f} degrees"
)

print(
    "\n=============================================="
)

print(
    "Motion model-based walking animation "
    "completed successfully."
)

print(
    "=============================================="
)


WALKING CHARACTER RIG RESULTS

Walking speed parameter: 1.20 units/s
Total walking displacement: 4.80 units
Total animation duration: 4.00 seconds
Estimated average speed: 1.20 units/s

Joint angle ranges:
Left hip: 50.00 degrees
Right hip: 50.00 degrees
Left knee: 25.00 degrees
Right knee: 25.00 degrees

Motion model-based walking animation completed successfully.
